# Step 6: EEG Quality Control and BrainLat Channel Mapping

This notebook performs:
1. Quality control on all preprocessed recordings
2. BrainLat 128-channel mapping to standard 19-channel 10-20 montage
3. Common channel output assessment
4. Visual quality check plots
5. Final decision on BrainLat inclusion

In [1]:
%pip install --force-reinstall "numpy==1.24.4" "pandas==1.5.3" "mne==1.6.1" "matplotlib==3.7.2" "scipy==1.10.1" -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
anaconda-project 0.11.1 requires ruamel-yaml, which is not installed.
pymatreader 1.1.0 requires scipy>=1.13.1, but you have scipy 1.10.1 which is incompatible.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.24.4 which is incompatible.
conda-repo-cli 1.0.20 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.20 requires nbformat==5.4.0, but you have nbformat 5.5.0 which is incompatible.
conda-repo-cli 1.0.20 requires python-dateutil==2.8.2, but you have python-dateutil 2.9.0.post0 which is incompatible.
conda-repo-cli 1.0.20 requires requests==2.28.1, but you have requests 2.32.5 which is incompatible.
botocore 1.27.28 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.6.3 which is incompati

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 200)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'NOTEBOOK':
    NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'NOTEBOOK' else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / 'processed_data'
CLEANED_ROOT = PROCESSED_DIR / 'cleaned_eeg'
QC_DIR = PROCESSED_DIR / 'quality_control'
REPORT_PATH = PROCESSED_DIR / 'preprocessing_report.csv'
SUMMARY_PATH = PROCESSED_DIR / 'preprocessing_summary.csv'
QC_REPORT_PATH = PROCESSED_DIR / 'quality_control_report.csv'
BRAINLAT_MAPPING_PATH = PROCESSED_DIR / 'brainlat_channel_mapping.csv'
FINAL_COMPARISON_PATH = PROCESSED_DIR / 'final_channel_comparison.csv'

print('MNE version:', mne.__version__)

MNE version: 1.6.1


## PART 1 — QUALITY CONTROL

Check every processed recording for:
- Correct sampling rate (250 Hz)
- Correct channel count (19)
- Correct channel names (standard 10-20)
- Missing values (NaN)
- Inf values
- Flat channels (std < 1 µV)
- Noisy channels (variance > 10× 99th percentile)
- Abnormal amplitude (abs > 1000 µV)
- Recording duration
- Preprocessing status
- Subject ID and diagnosis label

In [3]:
preproc_report = pd.read_csv(REPORT_PATH)
preproc_summary = pd.read_csv(SUMMARY_PATH)

print('Preprocessing report shape:', preproc_report.shape)
print('Status counts:')
print(preproc_report['status'].value_counts())
print('\nSummary:')
print(preproc_summary)

Preprocessing report shape: (310, 16)
Status counts:
success                     211
insufficient_channels        55
error: FileNotFoundError     44
Name: status, dtype: int64

Summary:
       Dataset  Total recordings  Successfully processed  Flagged recordings  \
0        ADFTD               176                     176                   0   
1     BrainLat                99                       0                  55   
2  AD_Auditory                35                      35                   0   

   Failed recordings Common sampling rate  Common channel count  \
0                  0               250 Hz                  19.0   
1                 44               250 Hz                   NaN   
2                  0               250 Hz                  19.0   

                Filter Notch frequency          Reference  
0  0.5-45 Hz band-pass         50.0 Hz  Average reference  
1  0.5-45 Hz band-pass         60.0 Hz  Average reference  
2  0.5-45 Hz band-pass         50.0 Hz  Aver

In [4]:
TARGET_CHANNELS = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8',
    'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3',
    'Pz', 'P4', 'T6', 'O1', 'O2'
]

THRESHOLDS = {
    'target_sfreq': 250.0,
    'target_n_channels': 19,
    'flat_threshold_uv': 1.0,
    'noisy_multiplier': 10.0,
    'amplitude_threshold_uv': 1000.0,
    'min_duration_sec': 30.0,
    'max_nan_fraction': 0.0,
}

print('QUALITY THRESHOLDS:')
for k, v in THRESHOLDS.items():
    print(f'  {k}: {v}')

QUALITY THRESHOLDS:
  target_sfreq: 250.0
  target_n_channels: 19
  flat_threshold_uv: 1.0
  noisy_multiplier: 10.0
  amplitude_threshold_uv: 1000.0
  min_duration_sec: 30.0
  max_nan_fraction: 0.0


In [5]:
def quality_check_recording(row):
    dataset = row['dataset']
    subject_id = row['subject_id']
    standard_label = row['standard_label']
    file_path = row['processed_file']
    
    result = {
        'dataset': dataset,
        'subject_id': subject_id,
        'standard_label': standard_label,
        'file_path': file_path,
        'sampling_rate': np.nan,
        'channel_count': np.nan,
        'duration': np.nan,
        'nan_count': 0,
        'inf_count': 0,
        'flat_channels': '',
        'noisy_channels': '',
        'max_amplitude_uv': np.nan,
        'quality_status': 'fail',
        'quality_reason': ''
    }
    
    reasons = []
    
    try:
        raw = mne.io.read_raw_fif(file_path, preload=True, verbose='ERROR')
        
        result['sampling_rate'] = raw.info['sfreq']
        result['channel_count'] = len(raw.ch_names)
        result['duration'] = raw.n_times / raw.info['sfreq']
        
        data = raw.get_data()
        
        if not np.isclose(raw.info['sfreq'], THRESHOLDS['target_sfreq'], atol=1.0):
            reasons.append(f'sampling_rate_mismatch: {raw.info["sfreq"]} != {THRESHOLDS["target_sfreq"]}')
        
        if len(raw.ch_names) != THRESHOLDS['target_n_channels']:
            reasons.append(f'channel_count_mismatch: {len(raw.ch_names)} != {THRESHOLDS["target_n_channels"]}')
        
        missing_ch = [ch for ch in TARGET_CHANNELS if ch not in raw.ch_names]
        extra_ch = [ch for ch in raw.ch_names if ch not in TARGET_CHANNELS]
        if missing_ch:
            reasons.append(f'missing_channels: {missing_ch}')
        if extra_ch:
            reasons.append(f'extra_channels: {extra_ch}')
        
        nan_count = np.isnan(data).sum()
        result['nan_count'] = int(nan_count)
        if nan_count > THRESHOLDS['max_nan_fraction'] * data.size:
            reasons.append(f'nan_values: {nan_count}')
        
        inf_count = np.isinf(data).sum()
        result['inf_count'] = int(inf_count)
        if inf_count > 0:
            reasons.append(f'inf_values: {inf_count}')
        
        flat_chs = []
        for i, ch in enumerate(raw.ch_names):
            if np.std(data[i]) * 1e6 < THRESHOLDS['flat_threshold_uv']:
                flat_chs.append(ch)
        result['flat_channels'] = '; '.join(flat_chs) if flat_chs else ''
        if flat_chs:
            reasons.append(f'flat_channels: {flat_chs}')
        
        variances = np.var(data, axis=1)
        var_thresh = np.percentile(variances, 99) * THRESHOLDS['noisy_multiplier']
        noisy_chs = []
        for i, ch in enumerate(raw.ch_names):
            if variances[i] > var_thresh:
                noisy_chs.append(ch)
        result['noisy_channels'] = '; '.join(noisy_chs) if noisy_chs else ''
        if noisy_chs:
            reasons.append(f'noisy_channels: {noisy_chs}')
        
        max_amp = np.max(np.abs(data)) * 1e6
        result['max_amplitude_uv'] = float(max_amp)
        if max_amp > THRESHOLDS['amplitude_threshold_uv']:
            reasons.append(f'excessive_amplitude: {max_amp:.1f} µV')
        
        if raw.n_times / raw.info['sfreq'] < THRESHOLDS['min_duration_sec']:
            reasons.append(f'short_duration: {raw.n_times / raw.info["sfreq"]:.1f} sec')
        
        if row['status'] != 'success':
            reasons.append(f'preprocessing_status: {row["status"]}')
        
        if not reasons:
            result['quality_status'] = 'pass'
            result['quality_reason'] = 'all_checks_passed'
        else:
            result['quality_status'] = 'fail'
            result['quality_reason'] = '; '.join(reasons)
        
        raw.close()
        
    except Exception as exc:
        result['quality_status'] = 'error'
        result['quality_reason'] = f'{type(exc).__name__}: {str(exc)[:200]}'
    
    return result

In [6]:
success_df = preproc_report[preproc_report['status'] == 'success'].copy()
print(f'Checking {len(success_df)} successful recordings...')

qc_results = []
for idx, row in success_df.iterrows():
    if idx % 50 == 0:
        print(f'  {idx+1}/{len(success_df)}')
    result = quality_check_recording(row)
    qc_results.append(result)

other_df = preproc_report[preproc_report['status'] != 'success'].copy()
for idx, row in other_df.iterrows():
    result = {
        'dataset': row['dataset'],
        'subject_id': row['subject_id'],
        'standard_label': row['standard_label'],
        'file_path': row['processed_file'],
        'sampling_rate': np.nan,
        'channel_count': np.nan,
        'duration': np.nan,
        'nan_count': 0,
        'inf_count': 0,
        'flat_channels': '',
        'noisy_channels': '',
        'max_amplitude_uv': np.nan,
        'quality_status': 'fail',
        'quality_reason': f'preprocessing_status: {row["status"]}; {row["flags"]}'
    }
    qc_results.append(result)

qc_report = pd.DataFrame(qc_results)
print(f'\nTotal QC records: {len(qc_report)}')
print('Quality status:')
print(qc_report['quality_status'].value_counts())

qc_report.to_csv(QC_REPORT_PATH, index=False)
print(f'Saved: {QC_REPORT_PATH}')

Checking 211 successful recordings...
  1/211
  51/211
  101/211
  151/211
  201/211

Total QC records: 310
Quality status:
fail    165
pass    145
Name: quality_status, dtype: int64
Saved: /home/admin1/Desktop/SIH_2026_ALL/processed_data/quality_control_report.csv


## PART 2 — BRAINLAT CHANNEL MAPPING

BrainLat uses 128 channels with numeric names (A1-D32) and has electrode coordinates.
We map to standard 19-channel 10-20 using nearest-neighbor matching on 3D positions.

In [7]:
bl_raw = mne.io.read_raw_fif('/home/admin1/Desktop/SIH_2026_ALL/processed_data/resampled/BrainLat/sub-20001/s6_sub-20001_rs-HEP_eeg_sr250.fif', preload=False, verbose='ERROR')
bl_pos = bl_raw.get_montage().get_positions()['ch_pos']
bl_raw.close()

std_montage = mne.channels.make_standard_montage('standard_1020')
std_pos = std_montage.get_positions()['ch_pos']

mapping_rows = []
used_bl_channels = set()

for target in TARGET_CHANNELS:
    if target not in std_pos:
        mapping_rows.append({
            'original_channel': '',
            'standard_channel': target,
            'mapping_method': 'not_in_standard_montage',
            'mapping_status': 'unavailable',
            'confidence_or_reason': 'Target not in standard 10-20 montage'
        })
        continue
    
    tpos = np.array(std_pos[target])
    best_ch = None
    best_dist = float('inf')
    
    for bl_ch, bpos in bl_pos.items():
        if bl_ch in used_bl_channels:
            continue
        dist = np.linalg.norm(tpos - np.array(bpos))
        if dist < best_dist:
            best_dist = dist
            best_ch = bl_ch
    
    if best_ch is None:
        mapping_rows.append({
            'original_channel': '',
            'standard_channel': target,
            'mapping_method': 'nearest_neighbor',
            'mapping_status': 'no_available_channel',
            'confidence_or_reason': 'All BrainLat channels already assigned'
        })
    else:
        used_bl_channels.add(best_ch)
        if best_dist < 0.01:
            confidence = 'high'
        elif best_dist < 0.02:
            confidence = 'medium'
        elif best_dist < 0.03:
            confidence = 'low'
        else:
            confidence = 'very_low'
        
        mapping_rows.append({
            'original_channel': best_ch,
            'standard_channel': target,
            'mapping_method': 'nearest_neighbor_3d',
            'mapping_status': 'mapped',
            'confidence_or_reason': f'{confidence} (distance={best_dist:.4f} m)'
        })

brainlat_mapping = pd.DataFrame(mapping_rows)
print('BRAINLAT CHANNEL MAPPING:')
print(brainlat_mapping.to_string(index=False))

BRAINLAT CHANNEL MAPPING:
original_channel standard_channel      mapping_method mapping_status         confidence_or_reason
             C29              Fp1 nearest_neighbor_3d         mapped     high (distance=0.0047 m)
             C16              Fp2 nearest_neighbor_3d         mapped     high (distance=0.0052 m)
              D7               F7 nearest_neighbor_3d         mapped   medium (distance=0.0123 m)
              D4               F3 nearest_neighbor_3d         mapped     high (distance=0.0053 m)
             C21               Fz nearest_neighbor_3d         mapped     high (distance=0.0072 m)
              C4               F4 nearest_neighbor_3d         mapped     high (distance=0.0066 m)
              C7               F8 nearest_neighbor_3d         mapped   medium (distance=0.0116 m)
             D24               T3 nearest_neighbor_3d         mapped   medium (distance=0.0127 m)
             D19               C3 nearest_neighbor_3d         mapped   medium (distance=0.01

In [8]:
brainlat_mapping.to_csv(BRAINLAT_MAPPING_PATH, index=False)
print(f'Saved: {BRAINLAT_MAPPING_PATH}')

Saved: /home/admin1/Desktop/SIH_2026_ALL/processed_data/brainlat_channel_mapping.csv


## PART 3 — COMMON CHANNEL OUTPUT

In [9]:
comparison_rows = []

comparison_rows.append({
    'Dataset': 'ADFTD',
    'Original Channel Count': 19,
    'Target Channel Count': 19,
    'Target Channels Available': 19,
    'Missing Target Channels': 0,
    'Mapping Status': 'Exact match (standard 10-20)'
})

comparison_rows.append({
    'Dataset': 'AD-Auditory',
    'Original Channel Count': 19,
    'Target Channel Count': 19,
    'Target Channels Available': 19,
    'Missing Target Channels': 0,
    'Mapping Status': 'Exact match after T7/T8/P7/P8 -> T3/T4/T5/T6 alias'
})

mapped = brainlat_mapping[brainlat_mapping['mapping_status'] == 'mapped']
missing = brainlat_mapping[brainlat_mapping['mapping_status'] != 'mapped']
high_conf = mapped[mapped['confidence_or_reason'].str.contains('high')]
med_conf = mapped[mapped['confidence_or_reason'].str.contains('medium')]
low_conf = mapped[mapped['confidence_or_reason'].str.contains('low')]

comparison_rows.append({
    'Dataset': 'BrainLat',
    'Original Channel Count': 128,
    'Target Channel Count': 19,
    'Target Channels Available': len(mapped),
    'Missing Target Channels': len(missing),
    'Mapping Status': f'Mapped via 3D positions: {len(high_conf)} high, {len(med_conf)} medium, {len(low_conf)} low confidence'
})

final_comparison = pd.DataFrame(comparison_rows)
final_comparison.to_csv(FINAL_COMPARISON_PATH, index=False)
print('FINAL CHANNEL COMPARISON:')
print(final_comparison.to_string(index=False))

FINAL CHANNEL COMPARISON:
    Dataset  Original Channel Count  Target Channel Count  Target Channels Available  Missing Target Channels                                              Mapping Status
      ADFTD                      19                    19                         19                        0                                Exact match (standard 10-20)
AD-Auditory                      19                    19                         19                        0          Exact match after T7/T8/P7/P8 -> T3/T4/T5/T6 alias
   BrainLat                     128                    19                         19                        0 Mapped via 3D positions: 5 high, 7 medium, 7 low confidence


## PART 4 — VISUAL QUALITY CHECK

Generate plots for representative recordings from each dataset.

In [10]:
def plot_qc_figure(dataset, subject_id, file_path, qc_dir):
    try:
        raw = mne.io.read_raw_fif(file_path, preload=True, verbose='ERROR')
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(f'{dataset} | {subject_id} | {raw.info["sfreq"]:.0f} Hz | {len(raw.ch_names)} ch', fontsize=14)
        
        raw.plot(duration=10, n_channels=19, axes=axes[0, 0], show=False, verbose='ERROR')
        axes[0, 0].set_title('EEG (first 10s, all 19 channels)')
        
        axes[0, 1].axis('off')
        ch_text = '\n'.join([f'{i+1:2d}. {ch}' for i, ch in enumerate(raw.ch_names)])
        axes[0, 1].text(0.1, 0.9, f'Channel Names ({len(raw.ch_names)}):\n{ch_text}', 
                       transform=axes[0, 1].transAxes, fontsize=10, fontfamily='monospace',
                       verticalalignment='top')
        axes[0, 1].set_title('Channel Names')
        
        raw.compute_psd(fmax=60).plot(axes=axes[1, 0], show=False, verbose='ERROR')
        axes[1, 0].axvline(0.5, color='r', linestyle='--', alpha=0.5, label='0.5 Hz')
        axes[1, 0].axvline(45, color='r', linestyle='--', alpha=0.5, label='45 Hz')
        axes[1, 0].axvline(50 if dataset == 'ADFTD' else 60, color='orange', linestyle=':', alpha=0.7, label='Notch')
        axes[1, 0].legend()
        axes[1, 0].set_title('Power Spectral Density')
        
        data = raw.get_data() * 1e6
        for i, ch in enumerate(raw.ch_names):
            axes[1, 1].hist(data[i].flatten(), bins=100, alpha=0.5, label=ch, density=True, histtype='step')
        axes[1, 1].set_xlabel('Amplitude (µV)')
        axes[1, 1].set_ylabel('Density')
        axes[1, 1].set_title('Amplitude Distribution')
        axes[1, 1].set_xlim(-500, 500)
        axes[1, 1].legend(ncol=3, fontsize=6)
        
        plt.tight_layout()
        output_fig = qc_dir / dataset / f'{subject_id}_qc_detailed.png'
        output_fig.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_fig, dpi=150, bbox_inches='tight')
        plt.close(fig)
        
        raw.close()
        
    except Exception as exc:
        print(f'QC plot failed for {dataset}/{subject_id}: {exc}')

In [11]:
pass_df = qc_report[qc_report['quality_status'] == 'pass']

for dataset in ['ADFTD', 'BrainLat', 'AD_Auditory']:
    if dataset == 'BrainLat':
        subset = preproc_report[preproc_report['dataset'] == 'BrainLat'].head(3)
        for _, row in subset.iterrows():
            if pd.notna(row.get('resampled_file')) and Path(row['resampled_file']).exists():
                plot_qc_figure(dataset, row['subject_id'], row['resampled_file'], QC_DIR)
    else:
        subset = pass_df[pass_df['dataset'] == dataset].head(3)
        for _, row in subset.iterrows():
            plot_qc_figure(dataset, row['subject_id'], row['file_path'], QC_DIR)
    print(f'QC plots generated for {dataset}')

QC plot failed for ADFTD/sub-001: plot() got an unexpected keyword argument 'axes'
QC plot failed for ADFTD/sub-001: plot() got an unexpected keyword argument 'axes'
QC plot failed for ADFTD/sub-002: plot() got an unexpected keyword argument 'axes'
QC plots generated for ADFTD
QC plots generated for BrainLat
QC plot failed for AD_Auditory/sub-01: plot() got an unexpected keyword argument 'axes'
QC plot failed for AD_Auditory/sub-03: plot() got an unexpected keyword argument 'axes'
QC plot failed for AD_Auditory/sub-04: plot() got an unexpected keyword argument 'axes'
QC plots generated for AD_Auditory


## PART 5 — DECISION

In [12]:
print('='*60)
print('FINAL DECISION REPORT')
print('='*60)

print('\n1. BRAINLAT MAPPING TO COMMON 19-CHANNEL MONTAGE:')
mapped = brainlat_mapping[brainlat_mapping['mapping_status'] == 'mapped']
print(f'   Mapped channels: {len(mapped)}/19')
print(f'   Unmapped channels: {len(brainlat_mapping) - len(mapped)}')

print('\n2. EXACT MATCHES (distance < 1 cm):')
high = mapped[mapped['confidence_or_reason'].str.contains('high')]
for _, r in high.iterrows():
    print(f'   {r["standard_channel"]:4s} <- {r["original_channel"]:4s}  {r["confidence_or_reason"]}')

print('\n3. ALIAS MAPPINGS (T7/T8/P7/P8 equivalents):')
for _, r in mapped.iterrows():
    if 'T3' in r['standard_channel'] or 'T4' in r['standard_channel'] or 'T5' in r['standard_channel'] or 'T6' in r['standard_channel']:
        print(f'   {r["standard_channel"]:4s} <- {r["original_channel"]:4s}  {r["confidence_or_reason"]}')

print('\n4. LOW CONFIDENCE / UNAVAILABLE:')
low = mapped[mapped['confidence_or_reason'].str.contains('low|very_low')]
for _, r in low.iterrows():
    print(f'   {r["standard_channel"]:4s} <- {r["original_channel"]:4s}  {r["confidence_or_reason"]}')
for _, r in missing.iterrows():
    print(f'   {r["standard_channel"]:4s} <- UNAVAILABLE  {r["confidence_or_reason"]}')

print('\n5. RECORDINGS TO EXCLUDE:')
fail_df = qc_report[qc_report['quality_status'] != 'pass']
print(f'   Total recordings: {len(qc_report)}')
print(f'   Pass QC: {len(qc_report) - len(fail_df)}')
print(f'   Fail QC: {len(fail_df)}')
for _, r in fail_df.iterrows():
    print(f'   {r["dataset"]}/{r["subject_id"]}: {r["quality_reason"][:100]}')

print('\n6. RECOMMENDATION:')
if len(mapped) >= 16:
    print('   BrainLat CAN be included with mapped 19-channel montage.')
    print('   Use position-based mapping for all 19 channels.')
    print('   Note: T5, P3, Pz, P4, T6, O1, O2 have lower confidence (>2 cm).')
else:
    print('   BrainLat CANNOT be reliably mapped to 19-channel montage.')

print('\nQUALITY CONTROL COMPLETE — READY FOR EEG SEGMENTATION')

FINAL DECISION REPORT

1. BRAINLAT MAPPING TO COMMON 19-CHANNEL MONTAGE:
   Mapped channels: 19/19
   Unmapped channels: 0

2. EXACT MATCHES (distance < 1 cm):
   Fp1  <- C29   high (distance=0.0047 m)
   Fp2  <- C16   high (distance=0.0052 m)
   F3   <- D4    high (distance=0.0053 m)
   Fz   <- C21   high (distance=0.0072 m)
   F4   <- C4    high (distance=0.0066 m)

3. ALIAS MAPPINGS (T7/T8/P7/P8 equivalents):
   T3   <- D24   medium (distance=0.0127 m)
   T4   <- B14   medium (distance=0.0138 m)
   T5   <- A10   low (distance=0.0212 m)
   T6   <- B7    low (distance=0.0218 m)

4. LOW CONFIDENCE / UNAVAILABLE:
   T5   <- A10   low (distance=0.0212 m)
   P3   <- A7    low (distance=0.0280 m)
   Pz   <- A19   low (distance=0.0284 m)
   P4   <- B4    low (distance=0.0282 m)
   T6   <- B7    low (distance=0.0218 m)
   O1   <- A16   very_low (distance=0.0310 m)
   O2   <- A29   very_low (distance=0.0307 m)

5. RECORDINGS TO EXCLUDE:
   Total recordings: 310
   Pass QC: 145
   Fail QC: 165